In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder


def preprocess_youth_dataset(filepath, model_type='rf'):
    """
    Parameters:
    - filepath (str): CSV 파일 경로
    - model_type (str): 'lr' (로지스틱 회귀 - 스케일러 적용) 또는 'rf' (랜덤 포레스트 - 스케일러 생략)

    Returns:
    - X_final: 전처리된 feature 데이터
    - y: target 데이터

    수정 사항:
    1. K-fold를 모델링 단계에서 사용하기 위해 train/test split 제거했습니다
    2. Logistic Regression scaling 시 One-Hot Encoding 결과 컬럼은 scaling 대상에서 제외했습니다
    """

    # 1. 데이터 로드
    df = pd.read_csv(filepath, encoding='cp949')
    df.columns = df.columns.str.strip()

    # 2. 타겟(Target) 변수 처리 및 불필요 컬럼 제거
    # 우울증상 유병 여부 (1: 없음 -> 0, 2: 있음 -> 1)
    df['Target'] = df['우울증상 유병 여부'].map({1: 0, 2: 1})
    y = df['Target']

    # 번아웃 컬럼 제거
    drop_cols = ['우울증상 유병 여부', 'Target', '최근 1년 소진(번아웃) 경험 여부']
    X = df.drop(columns=[col for col in drop_cols if col in df.columns])
    X = X.copy()

    # 3. '현재 근무 여부' 파생변수 생성
    # 경제활동상태의 값이 1(취업)인 경우에만 1(근무), 나머지는 0(비근무)
    if '경제활동상태' in X.columns:
        X['현재 근무 여부'] = X['경제활동상태'].apply(lambda x: 1 if x == 1 else 0)
    else:
        X['현재 근무 여부'] = 0

    # =========================================================================
    # 4. Ordinal 변수 역순 매핑 및 결측치 0 처리
    # 빈도/만족도가 높을수록 큰 값을 가지게 처리
    # =========================================================================

    # 논리 통일을 위해 조정이 필요한 컬럼
    ordinal_mappings = {
        '현재 흡연 여부': {4: 1, 3: 2, 2: 3, 1: 4},
        '최근 1년간 음주 빈도': {7: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6},
        '외식 또는 매식 빈도': {5: 0, 4: 1, 3: 2, 2: 3, 1: 4},
        '일자리 불안정성 정도 - 현재의 일을 그만두거나 실직하더라도 나는 비슷한 돈을 벌 수 있는 직업을 쉽게 찾을 수 있을 것이다': {
            5: 1, 4: 2, 3: 3, 2: 4, 1: 5
        },
        '정치에 대한 관심 정도': {4: 1, 3: 2, 2: 3, 1: 4},
        '외출 빈도': {8: 1, 7: 2, 6: 3, 5: 4, 4: 5, 3: 6, 2: 7, 1: 8},
        '주관적 계층 인식': {5: 1, 4: 2, 3: 3, 2: 4, 1: 5}
    }

    standard_ordinal_cols = [
        '연령별', '가구원수', '최종학력', '음주 정도', '평소 규칙적 운동 여부',
        '고용 계약 기간', '총 고용 예상 기간', '지난 주 일자리 종사자 수', '재직 기간(범위)'
    ]

    def apply_ordinal_transform(data):
        data = data.copy()

        # 정규근로시간 외 추가 근무
        # 0 = 비근무/결측, 1 = 해당없음, 이후 추가근무 정도가 커질수록 값 증가
        if '정규근로시간 외 추가 근무' in data.columns:
            data['정규근로시간 외 추가 근무'] = data['정규근로시간 외 추가 근무'].map(
                {6: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 7: 1} #모름은 극소수이기에 최빈값 없음으로 대체
            ).fillna(0)

        # 정규 근로일 외 휴일 근무 횟수
        # 0 = 비근무/결측, 1 = 실제 0회, 이후 휴일근무 횟수 증가
        if '정규 근로일 외 휴일 근무 횟수' in data.columns:
            data['정규 근로일 외 휴일 근무 횟수'] = data['정규 근로일 외 휴일 근무 횟수'].map(
                {0: 1, 1: 2, 2: 3}
            ).fillna(0)

        # 지정된 역순 매핑 적용
        for col, mapping in ordinal_mappings.items():
            if col in data.columns:
                data[col] = data[col].fillna(0)
                data[col] = data[col].map(mapping).fillna(0)

        # 나머지 ordinal 변수 결측치 0 처리
        for col in standard_ordinal_cols:
            if col in data.columns:
                data[col] = data[col].fillna(0)

        return data

    X = apply_ordinal_transform(X)

    # =========================================================================
    # 5. One-Hot Encoding 처리
    # =========================================================================

    nominal_cols = [
        '성별', '지역별', '장애여부','귀하의 현재 재학 상태를 응답해 주십시오.', '(복수 응답) 가구유형(1)',
        '국민기초생활보장제도(또는 맞춤형 급여) 수급 여부 및 경험', '돌봄 필요 가구원 유무', '혼인 상태',
        '부모 동거 여부', '공공 임대 주택 거주 경험', '현재 주거 점유 형태', '최근 1달 동안 같이 식사한 사람',
        '경제활동상태', '지난 주 수입을 목적으로 1시간 이상 일한 경험', '지난 주 종사상 지위',
        '고용 계약 기간 유무', '주휴 수당 수급 여부', '정규근로시간 외 추가 근무 시 추가 수당 수급 여부',
        '정규근로일 외 휴일 근무 시 추가 수당 수급 여부',
        '장시간 근로 경험(퇴근한 후부터 다음 날 출근하기까지의 시간이 11시간 미만)',
        '대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 - 가족^ 친척',
        '대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 – 이외',
        '현재 연애 여부(유배우 포함)', '향후 결혼 계획(유배우 포함)',
        '금융 채무 불이행자(신용불량자)에 해당 여부',
        '현재 근무 여부'
    ]

    valid_nominal_cols = [col for col in nominal_cols if col in X.columns]
    print(f"[Debug] valid_nominal_cols: {valid_nominal_cols}")

    for col in valid_nominal_cols:
        X[col] = X[col].fillna('해당없음').astype(str)

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    X_ohe = pd.DataFrame(
        ohe.fit_transform(X[valid_nominal_cols]),
        columns=ohe.get_feature_names_out(valid_nominal_cols),
        index=X.index
    )

    # OHE 결과 컬럼명 저장
    ohe_cols = list(X_ohe.columns)

    # =========================================================================
    # 6. 연속형 변수 log 변환
    # =========================================================================

    robust_cols = ['청년 연간소득 - 총 소득', '청년 기준 부채 총액', '청년 기준 재산총액']
    standard_cols = ['월 평균 총생활비']

    for col in robust_cols + standard_cols:
        if col in X.columns:
            X[col] = X[col].fillna(0)
            X[col] = np.log1p(X[col])

    # =========================================================================
    # 7. 최종 테이블 병합
    # =========================================================================

    # Drop the original nominal columns from X, as they are now represented by X_ohe
    X_non_nominal = X.drop(columns=valid_nominal_cols, errors='ignore')

    X_final = pd.concat(
        [X_non_nominal, X_ohe],
        axis=1
    )

    # =========================================================================
    # 8. 모델 타입에 따른 scaling
    # =========================================================================

    if model_type == 'lr':
        # RobustScaler 적용 대상: 소득, 부채, 재산
        valid_robust_cols = [col for col in robust_cols if col in X_final.columns]

        if valid_robust_cols:
            r_scaler = RobustScaler()
            X_final[valid_robust_cols] = r_scaler.fit_transform(X_final[valid_robust_cols])

        # 수정 사항:
        # 기존에는 OHE 결과까지 StandardScaler를 적용했지만,
        # One-Hot Encoding 결과는 이미 0/1 값이므로 scaling 대상에서 제외
        valid_standard_cols = [
            col for col in standard_cols + standard_ordinal_cols + list(ordinal_mappings.keys())
            if col in X_final.columns and col not in valid_robust_cols and col not in ohe_cols
        ]

        if valid_standard_cols:
            s_scaler = StandardScaler()
            X_final[valid_standard_cols] = s_scaler.fit_transform(X_final[valid_standard_cols])

        print(">> [완료] 로지스틱 회귀용 scaling 적용")
        print(">> [수정] One-Hot Encoding 결과 컬럼은 scaling 대상에서 제외")

    elif model_type == 'rf':
        print(">> [완료] 랜덤 포레스트를 위해 별도의 수치 scaling 생략")

    else:
        raise ValueError("model_type은 'lr' 또는 'rf'만 사용할 수 있습니다.")

    print(f"최종 데이터 형태 - X: {X_final.shape}, y: {y.shape}")

    return X_final, y

## 실행 방법 (How to run)

아래 코드를 실행하여 로지스틱 회귀 및 랜덤 포레스트 모델을 위한 데이터를 전처리할 수 있습니다.
`2024_청년삶실태조사.csv` 파일은 현재 작업 디렉토리에 있어야 합니다.

In [4]:
# 1. 로지스틱 회귀(Logistic Regression) 모델용 데이터
# 파일 경로를 실제 파일 위치로 변경해주세요.
X_lr, y_lr = preprocess_youth_dataset('2024_청년삶실태조사.csv', model_type='lr')
print("\n로지스틱 회귀용 데이터 X_lr shape:", X_lr.shape)
print("로지스틱 회귀용 데이터 y_lr shape:", y_lr.shape)

[Debug] valid_nominal_cols: ['성별', '지역별', '장애여부', '귀하의 현재 재학 상태를 응답해 주십시오.', '(복수 응답) 가구유형(1)', '국민기초생활보장제도(또는 맞춤형 급여) 수급 여부 및 경험', '돌봄 필요 가구원 유무', '혼인 상태', '부모 동거 여부', '공공 임대 주택 거주 경험', '현재 주거 점유 형태', '최근 1달 동안 같이 식사한 사람', '경제활동상태', '지난 주 수입을 목적으로 1시간 이상 일한 경험', '지난 주 종사상 지위', '고용 계약 기간 유무', '주휴 수당 수급 여부', '정규근로시간 외 추가 근무 시 추가 수당 수급 여부', '정규근로일 외 휴일 근무 시 추가 수당 수급 여부', '장시간 근로 경험(퇴근한 후부터 다음 날 출근하기까지의 시간이 11시간 미만)', '대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 - 가족^ 친척', '현재 연애 여부(유배우 포함)', '향후 결혼 계획(유배우 포함)', '금융 채무 불이행자(신용불량자)에 해당 여부', '현재 근무 여부']
>> [완료] 로지스틱 회귀용 scaling 적용
>> [수정] One-Hot Encoding 결과 컬럼은 scaling 대상에서 제외
최종 데이터 형태 - X: (15098, 124), y: (15098,)

로지스틱 회귀용 데이터 X_lr shape: (15098, 124)
로지스틱 회귀용 데이터 y_lr shape: (15098,)


In [5]:
# 2. 랜덤 포레스트(Random Forest) 모델용 데이터
# 파일 경로를 실제 파일 위치로 변경해주세요.
X_rf, y_rf = preprocess_youth_dataset('2024_청년삶실태조사.csv', model_type='rf')
print("\n랜덤 포레스트용 데이터 X_rf shape:", X_rf.shape)
print("랜덤 포레스트용 데이터 y_rf shape:", y_rf.shape)

[Debug] valid_nominal_cols: ['성별', '지역별', '장애여부', '귀하의 현재 재학 상태를 응답해 주십시오.', '(복수 응답) 가구유형(1)', '국민기초생활보장제도(또는 맞춤형 급여) 수급 여부 및 경험', '돌봄 필요 가구원 유무', '혼인 상태', '부모 동거 여부', '공공 임대 주택 거주 경험', '현재 주거 점유 형태', '최근 1달 동안 같이 식사한 사람', '경제활동상태', '지난 주 수입을 목적으로 1시간 이상 일한 경험', '지난 주 종사상 지위', '고용 계약 기간 유무', '주휴 수당 수급 여부', '정규근로시간 외 추가 근무 시 추가 수당 수급 여부', '정규근로일 외 휴일 근무 시 추가 수당 수급 여부', '장시간 근로 경험(퇴근한 후부터 다음 날 출근하기까지의 시간이 11시간 미만)', '대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 - 가족^ 친척', '현재 연애 여부(유배우 포함)', '향후 결혼 계획(유배우 포함)', '금융 채무 불이행자(신용불량자)에 해당 여부', '현재 근무 여부']
>> [완료] 랜덤 포레스트를 위해 별도의 수치 scaling 생략
최종 데이터 형태 - X: (15098, 124), y: (15098,)

랜덤 포레스트용 데이터 X_rf shape: (15098, 124)
랜덤 포레스트용 데이터 y_rf shape: (15098,)


## 데이터 전처리 결과 확인

아래 셀을 실행하여 전처리된 데이터의 결측치 및 형태를 확인합니다.

In [6]:
import pandas as pd
from IPython.display import display

# 출력 옵션 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.2f}'.format)



# ===========================
# Logistic Regression 데이터 확인
# ===========================

print("========== Logistic Regression ==========")

print("\n[데이터 크기]")
print("X shape:", X_lr.shape)
print("y shape:", y_lr.shape)

print("\n[y 분포]")
display(y_lr.value_counts())


print("\n[전처리 확인용 (transpose)]")
display(X_lr.iloc[:5, :20].T)

print("\n[결측치 확인]")
display(
    X_lr.isnull().sum().sort_values(ascending=False).head(20)
)

print("\n[dtype 확인]")
display(X_lr.dtypes)


# ===========================
# Random Forest 데이터 확인
# ===========================

print("\n\n========== Random Forest ==========")

print("\n[데이터 크기]")
print("X shape:", X_rf.shape)
print("y shape:", y_rf.shape)

print("\n[y 분포]")
display(y_rf.value_counts())

print("\n[상위 5개 row]")
display(X_rf.head())

print("\n[전처리 확인용 (transpose)]")
display(X_rf.iloc[:5, :20].T)

print("\n[결측치 확인]")
display(
    X_rf.isnull().sum().sort_values(ascending=False).head(100)
)

print("\n[dtype 확인]")
display(X_rf.dtypes)

========== Logistic Regression ==========

[데이터 크기]
X shape: (15098, 124)
y shape: (15098,)

[y 분포]


,count
Target,
0,13691
1,1407



[전처리 확인용 (transpose)]


,0,1,2,3,4
연령별,-1.16,-1.16,-1.16,-1.16,-1.16
가구원수,-1.23,-1.23,-1.23,-1.23,-1.23
최종학력,-1.91,-1.91,-1.91,-1.91,-1.91
현재 흡연 여부,2.00,1.13,2.00,2.00,2.00
최근 1년간 음주 빈도,-1.37,-0.04,0.63,0.63,0.63
음주 정도,-1.40,0.42,-0.19,0.42,1.63
평소 규칙적 운동 여부,1.72,-0.66,0.13,0.92,0.13
외식 또는 매식 빈도,-0.49,-0.49,-0.49,-0.49,-0.49
평소 본인에 대한 건강 인식,2.00,3.00,2.00,2.00,2.00
고용 계약 기간,-0.40,-0.40,-0.40,-0.40,-0.40



[결측치 확인]


...

,0,1,2,3,4
연령별,1.00,1.00,1.00,1.00,1.00
가구원수,1.00,1.00,1.00,1.00,1.00
최종학력,4.00,4.00,4.00,4.00,4.00
현재 흡연 여부,4.00,3.00,4.00,4.00,4.00
최근 1년간 음주 빈도,1.00,3.00,4.00,4.00,4.00
음주 정도,0.00,3.00,2.00,3.00,5.00
평소 규칙적 운동 여부,5.00,2.00,3.00,4.00,3.00
외식 또는 매식 빈도,1.00,1.00,1.00,1.00,1.00
평소 본인에 대한 건강 인식,2.00,3.00,2.00,2.00,2.00
고용 계약 기간,0.00,0.00,0.00,0.00,0.00



[결측치 확인]


,0
연령별,0
가구원수,0
최종학력,0
현재 흡연 여부,0
최근 1년간 음주 빈도,0
...,...
지난 주 종사상 지위_해당없음,0
고용 계약 기간 유무_1.0,0
고용 계약 기간 유무_2.0,0
고용 계약 기간 유무_해당없음,0



[dtype 확인]


,0
연령별,int64
가구원수,int64
최종학력,int64
현재 흡연 여부,int64
최근 1년간 음주 빈도,int64
...,...
향후 결혼 계획(유배우 포함)_3,float64
금융 채무 불이행자(신용불량자)에 해당 여부_1,float64
금융 채무 불이행자(신용불량자)에 해당 여부_2,float64
현재 근무 여부_0,float64
